# Lakebase Search Setup — Hybrid Vector + Full-Text
Enables BM25 full-text search (via `lakebase_text`) and vector similarity
(via `lakebase_vector` / pgvector) over product descriptions.

Data never leaves the Lakebase instance — retrieval is co-located with the operational data.

In [1]:
# Step 1: Install search extensions
# lakebase_vector installs pgvector automatically via CASCADE
# lakebase_text adds BM25 ranking (corpus-aware, top-K pushdown)
EXTENSIONS_SQL = '''
CREATE EXTENSION IF NOT EXISTS lakebase_vector CASCADE;
CREATE EXTENSION IF NOT EXISTS lakebase_text;
'''
print('Extensions to install:')
print('  - lakebase_vector (includes pgvector for embeddings)')
print('  - lakebase_text (BM25 ranking with top-K pushdown)')

Extensions to install:
  - lakebase_vector (includes pgvector for embeddings)
  - lakebase_text (BM25 ranking with top-K pushdown)


In [2]:
# Execute extension creation
import psycopg2
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()
token = w.tokens.create(comment='search-setup', lifetime_seconds=600).token_value

HOST = 'ep-little-union-d20vlyjv.database.us-east-1.cloud.databricks.com'
conn = psycopg2.connect(host=HOST, dbname='databricks_postgres',
    user='travis.lawrence@databricks.com', password=token, sslmode='require')
conn.autocommit = True
cur = conn.cursor()

cur.execute('CREATE EXTENSION IF NOT EXISTS lakebase_vector CASCADE')
print('✓ lakebase_vector + pgvector installed')

cur.execute('CREATE EXTENSION IF NOT EXISTS lakebase_text')
print('✓ lakebase_text installed')

cur.execute("SELECT extname FROM pg_extension ORDER BY extname")
print('\nInstalled extensions:')
for row in cur.fetchall():
    print(f'  {row[0]}')

✓ lakebase_vector + pgvector installed
✓ lakebase_text installed

Installed extensions:
  databricks_auth
  lakebase_text
  lakebase_vector
  neon
  plpgsql
  vector
  wal2delta


In [3]:
# Step 2: Create search table with tsvector column from product descriptions
cur.execute('''
CREATE TABLE IF NOT EXISTS northpeak_app.product_search AS
SELECT product_id, product_name, category, subcategory, description,
       to_tsvector('english',
           COALESCE(product_name, '') || ' ' ||
           COALESCE(description, '') || ' ' ||
           COALESCE(category, '') || ' ' ||
           COALESCE(subcategory, '')
       ) AS search_vector
FROM northpeak.synced_products
WHERE description IS NOT NULL
''')
print('✓ product_search table created from synced_products')

cur.execute('SELECT COUNT(*) FROM northpeak_app.product_search')
count = cur.fetchone()[0]
print(f'  {count} products indexed')

✓ product_search table created from synced_products
  1998 products indexed


In [4]:
# Step 3: Build BM25 index (must be AFTER data is populated)
cur.execute('''
CREATE INDEX IF NOT EXISTS idx_product_search_bm25
ON northpeak_app.product_search
USING lakebase_bm25 (search_vector)
''')
print('✓ BM25 index created: idx_product_search_bm25')
print('  Access method: lakebase_bm25')
print('  Column: search_vector (tsvector from product_name + description + category)')

✓ BM25 index created: idx_product_search_bm25
  Access method: lakebase_bm25
  Column: search_vector (tsvector from product_name + description + category)


In [5]:
# Step 4: Test natural-language query
NL_QUERY = 'warm winter parka insulated cold weather'
print(f'Natural language query: "{NL_QUERY}"\n')

cur.execute('''
SELECT product_id, product_name, description,
       search_vector <@> to_bm25query(
           to_tsvector('english', %s),
           'northpeak_app.idx_product_search_bm25'
       ) AS relevance_score
FROM northpeak_app.product_search
ORDER BY relevance_score
LIMIT 5
''', (NL_QUERY,))

results = cur.fetchall()
print(f"{'Rank':<5} {'Product ID':<15} {'Name':<30} {'Score':<10}")
print('-' * 65)
for i, (pid, name, desc, score) in enumerate(results, 1):
    print(f'{i:<5} {pid:<15} {name:<30} {score:.4f}')
    print(f'      {desc[:80]}')
    print()

cur.close()
conn.close()

Natural language query: "warm winter parka insulated cold weather"

Rank  Product ID      Name                           Score     
-----------------------------------------------------------------
1     SKU-APP-04412   Summit Down Parka               -19.1785
      Heavyweight insulated winter parka, 600-fill down, waterproof shell, storm hoo

2     SKU-APP-04418   Ridgeline Insulated Jacket      -18.0828
      Insulated winter jacket, synthetic fill, water-resistant shell — a warm midwei

3     SKU-APP-04460   Frostguard Thermal Gloves       -11.1349
      Insulated thermal winter gloves, touchscreen fingertips, water-resistant.

4     SKU-APP-10516   Tee 775                         -3.8921
      warm cold-weather footwear tee in apparel.

5     SKU-APP-10031   Shorts 911                      -3.8921
      warm cold-weather bottoms shorts in apparel.

